# How should a representative be chosen within each group?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [ ]:
import json

import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

pl.Config.set_tbl_rows(100)

def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]


In [ ]:
null_counts = [(c, df[c].null_count()) for c in v_cols]
df_nulls = pl.DataFrame(null_counts, schema=["column", "null_count"], orient="row")

family_sizes = (
    df_nulls.group_by("null_count")
    .agg(pl.len().alias("family_size"))
    .sort("family_size", descending=True)
)

fig = px.bar(family_sizes.to_pandas(), x="null_count", y="family_size", 
             title="Missingness Families: Size by Null Count",
             labels={"null_count": "Exact Null Count", "family_size": "Number of Columns in Family"},
             template="plotly_white")
fig.update_xaxes(type='category')
fig.show()

families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .sort("null_count", descending=True)
    .to_dict(as_series=False)
)

table = (
    "| Null count | Family size | Columns |\n"
    "| --- | ---: | --- |\n"
    + "\n".join(
        f"| {null_count} | {len(cols)} | {', '.join(cols)} |"
        for null_count, cols in zip(families["null_count"], families["cols"])
    )
)
display(Markdown(table))

missingness_families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .to_dict(as_series=False)
)
missingness_map = {c: cols for cols in missingness_families['cols'] for c in cols}


In [ ]:
def get_references_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "references").exists():
            return current / "references"
        current = current.parent
    return Path("../../references")

references_dir = get_references_dir()
with open(references_dir / "column-groups-v.json", "r") as f:
    col_groups_json = json.load(f)

subgroups = []
assigned_cols = set()
for block in col_groups_json['blocks']:
    for group in block['groups']:
        subgroups.append(group)
        assigned_cols.update(group)

unassigned = set(v_cols) - assigned_cols
for u in unassigned:
    subgroups.append([u])

table = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Total columns | {len(v_cols)} |\n"
    f"| Assigned in JSON | {len(assigned_cols)} |\n"
    f"| Unassigned | {len(unassigned)} |\n"
)
display(Markdown(table))


### Representative Selection

Selecting the column with the maximum `n_unique` from each sub-group as its representative.

In [ ]:
representatives = []
for group in subgroups:
    if len(group) == 1:
        representatives.append(group[0])
    else:
        uniques = [(c, df[c].n_unique()) for c in group]
        best_col = max(uniques, key=lambda x: x[1])[0]
        representatives.append(best_col)

total_cols = len(v_cols)
kept_cols = len(representatives)
dropped_cols = total_cols - kept_cols
reduction_pct = dropped_cols / total_cols * 100

fig = go.Figure(data=[go.Pie(labels=['Representatives Kept', 'Dropped'], 
                             values=[kept_cols, dropped_cols], hole=.4)])
fig.update_layout(title_text="Dimensionality Reduction via Selection", template="plotly_white")
fig.show()

table = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Columns in | {total_cols} |\n"
    f"| Representatives kept | **{kept_cols}** |\n"
    f"| Dropped | {dropped_cols} |\n"
    f"| Reduction | **{reduction_pct:.0f}%** |\n"
)
display(Markdown(table))
